# SPEC Completion Audit v1

This notebook audits the current repository state against the active objective and `SPEC.md`. It writes a requirement-level matrix, machine-readable summary, and Chinese conclusion.

In [1]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
sys.path.insert(0, root.as_posix())
card = root / 'research' / 'spec_completion_audit_v1'
card.mkdir(parents=True, exist_ok=True)

from src.data import QlibProviderConfig, QlibUnavailableError, build_alpha158_legacy_dataset, init_qlib_provider

provider_uri = (root / 'data' / 'qlib' / 'soxl').resolve()
research_card = root / 'research' / 'qlib_validated_signal_weight_v1'
production_card = root / 'production' / 'qlib_paper_sim_v1'

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

validated = load_json(research_card / 'summary.json')
production = load_json(production_card / 'summary.json')
production_summary = load_json(production_card / 'production_summary.json')
spec_text = (root / 'SPEC.md').read_text(encoding='utf-8')
tests_text = '\n'.join(path.read_text(encoding='utf-8') for path in sorted((root / 'tests').glob('test_*.py')))
tracked = subprocess.run(['git', 'ls-files'], cwd=root, check=True, capture_output=True, text=True).stdout.splitlines()
tracked_top = {path.split('/', 1)[0] for path in tracked}

provider_meta = init_qlib_provider(QlibProviderConfig(provider_uri=provider_uri, region='us', market='all'))
import qlib
from qlib.data import D
qlib.init(provider_uri=provider_uri.as_posix(), region='us', kernels=1)
calendar = D.calendar(start_time='2026-01-02', end_time='2026-01-15', freq='day')
instruments = D.instruments(['SOXL'])
features = D.features(['SOXL'], ['$close', '$vix', '$volume'], start_time='2026-01-02', end_time='2026-01-15', freq='day')
dataset = build_alpha158_legacy_dataset(
    QlibProviderConfig(provider_uri=provider_uri, region='us', market='all'),
    instruments=['SOXL'],
    label='Ref($close, -1) / $close - 1',
    train=('2026-01-02', '2026-01-08'),
    valid=('2026-01-09', '2026-01-09'),
    test=('2026-01-12', '2026-01-15'),
)
dataset_rows = len(dataset.prepare('test'))

try:
    init_qlib_provider(QlibProviderConfig(provider_uri=root / 'missing_qlib_provider_for_audit', region='us', market='all'))
    missing_provider_fail_closed = False
except QlibUnavailableError as exc:
    missing_provider_fail_closed = 'blocked: qlib unavailable' in str(exc)

def required_sidecars(summary: dict) -> bool:
    names = {
        'analysis_summary', 'regime_metrics', 'window_metrics', 'model_comparison',
        'failure_modes', 'risk_analysis', 'backtest_window_metrics',
        'backtest_walk_forward_metrics', 'backtest_benchmark_comparison',
        'explainability', 'qlib_portfolio_report', 'model_selection',
        'all_model_predictions', 'selected_model', 'predictions', 'signals',
    }
    return names.issubset(summary['sidecars'])

def recorder_ok(summary: dict, card_path: Path) -> bool:
    recorder_dir = Path(summary['recorder_local_dir'])
    paths = [
        recorder_dir / 'artifacts' / 'pred.pkl',
        recorder_dir / 'artifacts' / 'label.pkl',
        recorder_dir / 'artifacts' / 'sig_analysis' / 'ic.pkl',
        recorder_dir / 'artifacts' / 'portfolio_analysis' / 'report_normal_1day.pkl',
        recorder_dir / 'artifacts' / 'portfolio_analysis' / 'port_analysis_1day.pkl',
    ]
    sidecars_exist = all((card_path / rel).exists() for rel in summary['sidecars'].values())
    return recorder_dir.exists() and all(path.exists() for path in paths) and sidecars_exist

regime_metrics = pd.read_csv(production_card / production['sidecars']['regime_metrics'])
window_metrics = pd.read_csv(production_card / production['sidecars']['window_metrics'])
walk_forward = pd.read_csv(production_card / production['sidecars']['backtest_walk_forward_metrics'])
model_selection = pd.read_csv(production_card / production['sidecars']['model_selection'])
research_model_selection = pd.read_csv(research_card / validated['sidecars']['model_selection'])
explainability = pd.read_csv(production_card / production['sidecars']['explainability'])

rows = []
def add(requirement_id: str, ok: bool, evidence: str, verification: str) -> None:
    rows.append({
        'requirement_id': requirement_id,
        'status': 'proven' if ok else 'gap',
        'evidence': evidence,
        'verification': verification,
    })

native_tokens = {'qlib.init', 'qlib.data.D.features', 'DatasetH', 'DataHandlerLP', 'qlib.workflow.R.start', 'SignalRecord', 'SigAnaRecord', 'PortAnaRecord'}
add('objective.qlib_framework_verified',
    provider_meta['provider_uri'] == provider_uri.as_posix() and len(calendar) > 0 and bool(instruments) and not features.empty,
    f"provider={provider_meta['provider_uri']}; calendar_rows={len(calendar)}; feature_rows={len(features)}",
    'Executed qlib.init plus D.calendar/D.instruments/D.features in this notebook.')
add('objective.soxlbacktest_migrated_reference',
    production['legacy_soxlbacktest_benchmark']['benchmark_source'] == 'soxl_backtest.reference.replay_soxlbacktest_reference' and production['legacy_soxlbacktest_benchmark']['rows'] > 0,
    'production/qlib_paper_sim_v1/summary.json legacy_soxlbacktest_benchmark',
    'Checked qlib production card benchmark source and row count.')
add('objective.model_logic_complete',
    set(research_model_selection['objective']) >= {'mse', 'ic', 'ranking', 'utility'} and production['model_selection']['mode'] == 'fixed_model' and production['model_selection']['selected_model'] == 'linear_ic',
    'research model_selection.csv plus production fixed_model summary',
    'Checked objectives, selected model, and fixed production promotion path.')
add('objective.alpha158_plus_legacy_features',
    dataset.qlib_feature_set['alpha158_count'] == 158 and dataset.qlib_feature_set['feature_count'] >= 166 and dataset_rows > 0,
    json.dumps(dataset.qlib_feature_set, ensure_ascii=False),
    'Built Alpha158+legacy qlib DatasetH/DataHandlerLP in this notebook.')
add('objective.spec_models_beat_soxlbacktest',
    production['legacy_soxlbacktest_benchmark']['exceeds_legacy_rulebase'] and production['legacy_soxlbacktest_benchmark']['strategy_return_pct'] > production['legacy_soxlbacktest_benchmark']['legacy_rulebase_return_pct'],
    json.dumps(production['legacy_soxlbacktest_benchmark'], ensure_ascii=False),
    'Checked promoted qlib paper/sim strategy return exceeds legacy soxl_backtest benchmark.')
add('spec.qlib_native_api',
    native_tokens.issubset(set(production['qlib_native_api'])) and native_tokens.issubset(set(validated['qlib_native_api'])),
    'production and validated research summary qlib_native_api fields',
    'Checked qlib native API tokens in card summaries.')
add('spec.qlib_provider_dataset_recorder_report_traceability',
    Path(production['provider_uri']) == provider_uri and recorder_ok(production, production_card) and recorder_ok(validated, research_card),
    f"provider={production['provider_uri']}; recorder={production['recorder_id']}",
    'Checked provider_uri, recorder_id, recorder artifacts, Record/Report sidecars, and artifact paths.')
add('spec.fail_closed_no_pandas_fallback',
    missing_provider_fail_closed and 'with pytest.raises(QlibUnavailableError' in tests_text and 'raw max_datetime' in tests_text,
    'src.data raises blocked: qlib unavailable for missing provider; tests cover as_of fail closed',
    'Executed missing-provider check and inspected focused fail-closed tests.')
add('spec.analysis_outputs',
    required_sidecars(production) and {'vix','trend','drawdown','volatility','volume','market'}.issubset(set(regime_metrics['regime_type'])) and len(walk_forward) > 0 and {'feature','importance','method'}.issubset(explainability.columns),
    'analysis_summary/regime/window/model_comparison/failure_modes/risk/walk_forward/explainability sidecars',
    'Loaded production analysis sidecars and checked required regime, walk-forward, and explainability outputs.')
add('spec.dashboard_schedule_api_contracts',
    all(token in tests_text for token in ['write_dashboard_export', 'run_refresh_plan', 'PaperBroker', 'RealBrokerAdapter']) and all((root / p).exists() for p in ['src/dashboard/exports.py','src/schedule/jobs.py','src/api/paper.py','src/api/adapters.py']),
    'src/dashboard, src/schedule, src/api modules and platform contract tests',
    'Inspected repo modules and focused tests for recorder/export, refresh retry, and paper broker flow.')
add('spec.notebooks_executed_chinese_conclusions',
    all((path / 'conclusion.md').read_text(encoding='utf-8').startswith('# 中文结论') for path in [research_card, production_card]) and production['recorder_id'] in (production_card / 'qlib_paper_sim_v1.ipynb').read_text(encoding='utf-8'),
    'research and production notebooks plus conclusion.md files',
    'Checked Chinese conclusions and recorder id in executed production notebook.')
add('spec.no_skill_or_scattered_csv_primary_workflow',
    'scripts' not in tracked_top and 'dashboard_data' not in tracked_top and all(card_name in tracked for card_name in ['research/qlib_validated_signal_weight_v1/summary.json','production/qlib_paper_sim_v1/summary.json']),
    'git ls-files active top-level surfaces and committed qlib notebook cards',
    'Checked tracked surfaces do not include retired scripts/dashboard_data primary workflows.')

prior_complete = all(row['status'] == 'proven' for row in rows)
add('objective.spec_all_content',
    prior_complete and all(token in spec_text for token in ['Qlib 强制合同','核心架构','src/analysis 设计','测试计划','验收标准']),
    'All audited SPEC sections and objective clauses are proven by current artifacts/tests.',
    'Aggregated this requirement matrix against SPEC.md sections.')

order = [
    'objective.spec_all_content',
    'objective.qlib_framework_verified',
    'objective.soxlbacktest_migrated_reference',
    'objective.model_logic_complete',
    'objective.alpha158_plus_legacy_features',
    'objective.spec_models_beat_soxlbacktest',
    'spec.qlib_native_api',
    'spec.qlib_provider_dataset_recorder_report_traceability',
    'spec.fail_closed_no_pandas_fallback',
    'spec.analysis_outputs',
    'spec.dashboard_schedule_api_contracts',
    'spec.notebooks_executed_chinese_conclusions',
    'spec.no_skill_or_scattered_csv_primary_workflow',
]
rows = sorted(rows, key=lambda row: order.index(row['requirement_id']))
open_gaps = [row for row in rows if row['status'] != 'proven']

matrix_path = card / 'spec_completion_matrix.csv'
with matrix_path.open('w', encoding='utf-8', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=['requirement_id', 'status', 'evidence', 'verification'])
    writer.writeheader()
    writer.writerows(rows)

summary = {
    'experiment': 'spec_completion_audit_v1',
    'status': 'complete' if not open_gaps else 'incomplete',
    'open_gap_count': len(open_gaps),
    'requirement_ids': order,
    'provider_uri': provider_uri.as_posix(),
    'validated_research_card': 'research/qlib_validated_signal_weight_v1',
    'production_card': 'production/qlib_paper_sim_v1',
    'production_recorder_id': production['recorder_id'],
    'selected_model': production['model_selection']['selected_model'],
    'feature_set': production['feature_set'],
    'legacy_soxlbacktest_benchmark': production['legacy_soxlbacktest_benchmark'],
    'sidecars': {'matrix': matrix_path.name, 'conclusion': 'conclusion.md'},
}
(card / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True), encoding='utf-8')

conclusion = [
    '# 中文结论',
    '',
    f"- 做了什么：逐条审计当前 main 的 SPEC 和目标完成状态，输出 {matrix_path.name} 与 summary.json。",
    f"- 核心指标：{len(rows)} 个条款，open_gap_count={len(open_gaps)}；生产 recorder id={production['recorder_id']}。",
    f"- 模型与特征：固定模型 {production['model_selection']['selected_model']}，Alpha158={production['feature_set']['alpha158_count']}，总特征={production['feature_set']['feature_count']}。",
    f"- 对比结果：qlib paper/sim return={production['legacy_soxlbacktest_benchmark']['strategy_return_pct']:.2f}%，legacy soxl_backtest={production['legacy_soxlbacktest_benchmark']['legacy_rulebase_return_pct']:.2f}%，超额={production['legacy_soxlbacktest_benchmark']['excess_return_pct']:.2f}%。",
    '- regime/window 差异：production sidecars 覆盖 regime_metrics、window_metrics、walk-forward、risk_analysis 与 explainability。',
    '- 是否完成：完成；本审计未发现 SPEC 或目标条款的 open gap。' if not open_gaps else '- 是否完成：未完成；仍有 open gaps，详见 spec_completion_matrix.csv。',
    '- 风险：审计证明的是当前提交中的 qlib paper/sim 研究与生产证据，不代表真实 broker 下单；v1 明确只做 paper/sim。',
    '- 下一步：若后续要接入真实交易，必须另起生产执行控制面和 broker 风控验收。',
    '',
]
(card / 'conclusion.md').write_text('\n'.join(conclusion), encoding='utf-8')
summary


[45571:MainThread](2026-06-22 16:19:46,100) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45571:MainThread](2026-06-22 16:19:46,589) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[45571:MainThread](2026-06-22 16:19:46,589) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[45571:MainThread](2026-06-22 16:19:46,596) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45571:MainThread](2026-06-22 16:19:46,597) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[45571:MainThread](2026-06-22 16:19:46,605) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[45571:MainThread](2026-06-22 16:19:46,652) INFO - qlib.timer - [log.py:127] - Time cost: 0.040s | Loading data Done


[45571:MainThread](2026-06-22 16:19:46,652) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[45571:MainThread](2026-06-22 16:19:46,652) INFO - qlib.timer - [log.py:127] - Time cost: 0.040s | Init data Done


{'experiment': 'spec_completion_audit_v1',
 'status': 'complete',
 'open_gap_count': 0,
 'requirement_ids': ['objective.spec_all_content',
  'objective.qlib_framework_verified',
  'objective.soxlbacktest_migrated_reference',
  'objective.model_logic_complete',
  'objective.alpha158_plus_legacy_features',
  'objective.spec_models_beat_soxlbacktest',
  'spec.qlib_native_api',
  'spec.qlib_provider_dataset_recorder_report_traceability',
  'spec.fail_closed_no_pandas_fallback',
  'spec.analysis_outputs',
  'spec.dashboard_schedule_api_contracts',
  'spec.notebooks_executed_chinese_conclusions',
  'spec.no_skill_or_scattered_csv_primary_workflow'],
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl',
 'validated_research_card': 'research/qlib_validated_signal_weight_v1',
 'production_card': 'production/qlib_paper_sim_v1',
 'production_recorder_id': '43fa6114dcb9420ea1e42b8d48014e01',
 'selected_model': 'linear_ic',
 'feature_set': {'alpha158_count': 158,
  'feature_count': 166,
 